# Colab GPU runtime — CathAction unfreeze-backbone smoke test

Run this **in Google Colab** (Runtime > Change runtime type > GPU, T4 is plenty). Same SSH-tunnel-to-VS-Code setup as `colab_main.ipynb`, pointed at `config_cathaction.yaml` and the smoke-test command instead of a full ARCADE run.

Before running: push your local changes (unfrozen mit-b2 backbone, OOM safeguards, loss pos_weight, etc.) to the branch this clones below — the clone step pulls from GitHub, it does not see uncommitted local edits.

In [1]:
!nvidia-smi

Sat Aug  1 11:40:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# One-time setup: SSH server + cloudflared tunnel (no ngrok account needed, no flaky colab_ssh package)
!apt-get -qq -y install openssh-server > /dev/null
!mkdir -p /var/run/sshd
!echo "root:changeme" | chpasswd
!sed -i "s/#PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config
!sed -i "s/#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config
!service ssh restart

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

 * Restarting OpenBSD Secure Shell server sshd
   ...done.


In [3]:
import subprocess, time, re

# start tunnel in background, log to file so we can grab the printed URL
proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "ssh://localhost:22", "--logfile", "/content/cloudflared.log"]
)

url = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open("/content/cloudflared.log").read()
    except FileNotFoundError:
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+trycloudflare\.com", log)
    if match:
        url = match.group(0)
        break

if not url:
    raise RuntimeError("cloudflared didn't print a URL in time -- check /content/cloudflared.log")

hostname = url.replace("https://", "")
print(f"""Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName {hostname}
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h
""")

Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName florida-geneva-indicated-exact.trycloudflare.com
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h



The cell above prints an SSH `Host` block. It requires `cloudflared` installed **locally** too (the `ProxyCommand` runs it on your machine) — install it from https://github.com/cloudflare/cloudflared/releases or `winget install --id Cloudflare.cloudflared`.

Paste the block into your local `~/.ssh/config` (VS Code Command Palette > "Remote-SSH: Open SSH Configuration File"), save, then Command Palette > "Remote-SSH: Connect to Host" > `colab`. Enter password `changeme` when prompted.

In [4]:
# Get the repo onto the Colab VM -- make sure your local fix branch is pushed first
!git clone https://github.com/sormazabal/Phillips_UC2.git /content/Phillips_UC2

Cloning into '/content/Phillips_UC2'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (294/294), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 294 (delta 176), reused 223 (delta 105), pack-reused 0 (from 0)
Receiving objects: 100% (294/294), 148.78 KiB | 1.08 MiB/s, done.
Resolving deltas: 100% (176/176), done.


In [ ]:
%cd /content/Phillips_UC2
!pip install -q -r requirements.txt

/content/Phillips_UC2


## Dataset upload

`config_cathaction.yaml` only reads two things under `CathAction/` -- everything else (e.g. the `mask/` folder) is unused, since `src/data/dataset.py` rasterizes segmentation masks on the fly from the COCO `"segmentation"` polygons rather than reading mask images:

```
CathAction/coco/annotations/train.json
CathAction/coco/annotations/val.json
CathAction/coco/annotations/test.json
CathAction/segmentation_human_train/human_dataset_train/img/   (only the images referenced by the JSONs)
```

~168MB total. Upload that folder structure to Google Drive (e.g. `MyDrive/CathAction`), add a Drive shortcut if it's a shared folder, then set `DRIVE_DATASET_PATH` below.

In [27]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [28]:

# ponytail: shared folders only show up under MyDrive if you added a shortcut to them first
# (open the Drive link -> right-click the folder -> "Add shortcut to Drive"), then set the name below.
DRIVE_DATASET_PATH = '/content/drive/MyDrive/CathAction'
!ln -sfn "$DRIVE_DATASET_PATH" /content/Phillips_UC2/CathAction
!ls /content/Phillips_UC2/CathAction

coco  segmentation_human_train	segmentation_human_train.zip


In [22]:
# Load secrets from the .env file in Drive (HF_TOKEN, WANDB_API_KEY, etc.) --
# the key never appears in this notebook, only in the Drive file.
import os

env_path = '/content/drive/MyDrive/Env_vars/.env'
loaded = []
with open(env_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        key = key.strip()
        os.environ[key] = value.strip().strip('"').strip("'")
        loaded.append(key)

# No WANDB_API_KEY in the .env -> force offline so the training cell below never
# blocks on an interactive wandb login prompt (which Colab's !python can't answer).
if 'WANDB_KEY' in os.environ:
    os.environ['WANDB_API_KEY'] = os.environ['WANDB_KEY']
else:
    os.environ['WANDB_MODE'] = 'offline'

print(f"Loaded from {env_path}: {loaded}")
print(f"wandb mode: {'online (key found)' if 'WANDB_KEY' in os.environ else 'offline (no WANDB_KEY in .env)'}")

Loaded from /content/drive/MyDrive/Env_vars/.env: ['GITHUB_TOKEN', 'HF_TOKEN', 'WANDB_KEY']
wandb mode: online (key found)


## Smoke test

1 epoch, 20 train batches, 5 val batches -- confirms the unfrozen mit-b2 backbone fits in VRAM and trains without crashing before committing to the full 40-epoch run. Check the printed model summary for **`Trainable params` ~= 24-26M** (not ~2M), and that it completes without an OOM traceback.

In [23]:
%env PYTHONPATH=/content/Phillips_UC2

!python scripts/train.py --config config_cathaction.yaml --max_epochs 1 --limit_train_batches 20 --limit_val_batches 5 --checkpoint_dir /content/drive/MyDrive/CathAction_checkpoints

env: PYTHONPATH=/content/Phillips_UC2
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Loading weights: 100% 364/364 [00:00<00:00, 32900.76it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b2
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU availab

## Full run (only after the smoke test passes)

Baseline to beat: **val dice 0.4214** (peak-threshold dice measured on the old frozen-backbone checkpoint). Colab sessions can disconnect after ~12h/idle timeout -- if a 40-epoch run risks running long, either lower `max_epochs` per session and `--resume` from `checkpoints/last.ckpt`, or copy checkpoints to Drive periodically as `colab_main.ipynb`'s checkpoint-loading cell does for ARCADE.

In [ ]:
%env PYTHONPATH=/content/Phillips_UC2
!python scripts/train.py --config config_cathaction.yaml --checkpoint_dir /content/drive/MyDrive/CathAction_checkpoints

env: PYTHONPATH=/content/Phillips_UC2
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Loading weights: 100% 364/364 [00:00<00:00, 23959.18it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b2
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU availab